# 🚀 RAG Chapter 3 — Deploying to Production
## Production RAG Notebook: Databricks + Google Colab

This notebook implements the key concepts from Chapter 3 of *Building RAG Applications* (O'Reilly).

**Compatible with:**
- ✅ **Databricks** (recommended) — uses Delta Lake, Databricks Vector Search, MLflow, Structured Streaming
- ✅ **Google Colab** — uses local/GCS alternatives with the same RAG logic

**What you'll build:**
1. Delta Lake document store with Change Data Capture (CDC)
2. Production-grade ingestion pipeline with docling
3. RBAC-filtered hybrid search (BM25 + vector)
4. Two-stage retrieval with BGE-v2 reranking
5. Prompt engineering for production quality
6. Hallucination detection with HHEM scoring
7. MLflow experiment tracking for every query
8. Latency monitoring with P95 alerts
9. A/B testing framework for safe upgrades
10. Cost estimation utility

---

## 0. Environment Detection & Setup

In [ ]:
import sys, os

# Detect runtime environment
IS_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

print(f'Runtime: {"Databricks" if IS_DATABRICKS else "Google Colab" if IS_COLAB else "Local"}')
print(f'Python: {sys.version}')

In [ ]:
# Install dependencies (run once)
# On Databricks: attach cluster library or use %pip
# On Colab: run this cell

%pip install --quiet \
    docling \
    sentence-transformers \
    rank-bm25 \
    FlagEmbedding \
    mlflow \
    openai \
    weaviate-client \
    vectara-hhem \
    tqdm \
    pandas \
    pyarrow

# Databricks-specific (skip on Colab)
if IS_DATABRICKS:
    %pip install --quiet databricks-vectorsearch delta-spark

In [ ]:
# ─── Configuration ────────────────────────────────────────────
import os

# --- LLM Provider (set one) ---
LLM_PROVIDER    = 'openai'        # 'openai' | 'anthropic' | 'local'
OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY', 'sk-...')
LLM_MODEL       = 'gpt-4o-mini'   # or 'claude-3-5-haiku' for Anthropic

# --- Embedding ---
EMBED_MODEL     = 'BAAI/bge-base-en-v1.5'
RERANK_MODEL    = 'BAAI/bge-reranker-v2-m3'

# --- Vector Store ---
VS_BACKEND      = 'weaviate_local'  # 'weaviate_local' | 'databricks' | 'chroma'

# --- Databricks (only if IS_DATABRICKS) ---
DB_CATALOG      = 'main'
DB_SCHEMA       = 'rag_prod'
DB_TABLE_PATH   = f'/delta/{DB_CATALOG}/{DB_SCHEMA}/documents'
DB_VS_ENDPOINT  = 'rag-prod-endpoint'
DB_VS_INDEX     = f'{DB_CATALOG}.{DB_SCHEMA}.doc_index'

# --- RBAC Roles ---
ROLE_MAP = {
    'public':       ['public', 'faq'],
    'employee':     ['public', 'faq', 'internal'],
    'hr_manager':   ['public', 'faq', 'internal', 'hr'],
    'exec':         ['public', 'faq', 'internal', 'hr', 'finance', 'strategy'],
}

# --- Quality Thresholds ---
HHEM_PASS_THRESHOLD    = 0.75
HHEM_ABSTAIN_THRESHOLD = 0.60
LATENCY_P95_TARGET_MS  = 6000

print('Config loaded.')

## 1. Advanced Document Ingestion (docling + Delta Lake)

In [ ]:
import hashlib, json
from pathlib import Path
from docling.document_converter import DocumentConverter
from langchain.text_splitter import RecursiveCharacterTextSplitter

converter = DocumentConverter()
splitter  = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=['\n\n', '\n', '. ', ' ']
)

def load_and_chunk(file_path: str, collection: str = 'public') -> list[dict]:
    """
    Load a document with docling (preserves layout, tables, reading order)
    and split into overlapping chunks with metadata.
    Supports: PDF, DOCX, PPTX, HTML, XLSX
    """
    result   = converter.convert(file_path)
    markdown = result.document.export_to_markdown()
    chunks   = splitter.split_text(markdown)
    
    return [
        {
            'chunk_id':   hashlib.md5(f'{file_path}_{i}'.encode()).hexdigest(),
            'doc_id':     hashlib.md5(file_path.encode()).hexdigest(),
            'chunk_text': chunk,
            'chunk_idx':  i,
            'source':     file_path,
            'collection': collection,
            'hash':       hashlib.md5(chunk.encode()).hexdigest(),
            'total_chunks': len(chunks),
        }
        for i, chunk in enumerate(chunks)
    ]

# ── Demo: load a sample document ──────────────────────────────
# Replace with your actual file paths
SAMPLE_DOCS = [
    # ('path/to/user_manual.pdf',  'public'),
    # ('path/to/hr_policy.docx',   'hr'),
    # ('path/to/finance_q4.xlsx',  'finance'),
]

all_chunks = []
for path, collection in SAMPLE_DOCS:
    chunks = load_and_chunk(path, collection)
    all_chunks.extend(chunks)
    print(f'Loaded {path}: {len(chunks)} chunks')

print(f'Total chunks: {len(all_chunks)}')

In [ ]:
# ── Delta Lake CDC Upsert (Databricks) ────────────────────────
# On Colab: stores to a local pandas DataFrame + parquet fallback

import pandas as pd

def upsert_chunks(chunks: list[dict], table_name: str = 'documents'):
    """Upsert chunks to Delta Lake with CDC tracking (Databricks)
    or local parquet (Colab/local)."""
    
    if not chunks:
        print('No chunks to upsert.'); return
    
    df_new = pd.DataFrame(chunks)
    
    if IS_DATABRICKS:
        from delta.tables import DeltaTable
        spark_df = spark.createDataFrame(df_new)
        
        if DeltaTable.isDeltaTable(spark, DB_TABLE_PATH):
            dt = DeltaTable.forPath(spark, DB_TABLE_PATH)
            dt.alias('tgt').merge(
                spark_df.alias('src'),
                'tgt.chunk_id = src.chunk_id'
            ).whenMatchedUpdateAll() \
             .whenNotMatchedInsertAll() \
             .execute()
            print(f'Delta MERGE: {len(chunks)} chunks upserted.')
        else:
            spark_df.write.format('delta') \
                .option('delta.enableChangeDataFeed', 'true') \
                .save(DB_TABLE_PATH)
            spark.sql(f"""CREATE TABLE IF NOT EXISTS {DB_CATALOG}.{DB_SCHEMA}.{table_name}
                          USING DELTA LOCATION '{DB_TABLE_PATH}'""")
            print(f'Delta CREATE: {len(chunks)} chunks written.')
    else:
        # Colab / local: parquet
        parquet_path = f'/tmp/{table_name}.parquet'
        if os.path.exists(parquet_path):
            existing = pd.read_parquet(parquet_path)
            merged = pd.concat([existing, df_new]).drop_duplicates('chunk_id', keep='last')
        else:
            merged = df_new
        merged.to_parquet(parquet_path, index=False)
        print(f'Parquet upsert: {len(merged)} total chunks stored at {parquet_path}')

upsert_chunks(all_chunks)

## 2. Embedding + Vector Index Setup

In [ ]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import numpy as np

embed_model = SentenceTransformer(EMBED_MODEL)
print(f'Embed model loaded: {EMBED_MODEL}')

# ── In-memory index (Colab) or Databricks Vector Search ───────
# For production on Databricks, replace this with
# Databricks Vector Search TRIGGERED sync (see slide 13 code).

class InMemoryVectorIndex:
    """Simple in-memory HNSW-style index for Colab / testing.
    In production: replace with Databricks VS / Weaviate / Pinecone."""
    
    def __init__(self):
        self.chunks     = []
        self.embeddings = None
        self.bm25       = None
    
    def build(self, chunks: list[dict]):
        self.chunks = chunks
        texts = [c['chunk_text'] for c in chunks]
        
        print('Building vector index...')
        self.embeddings = embed_model.encode(texts, batch_size=64, show_progress_bar=True)
        
        print('Building BM25 index...')
        tokenized = [t.lower().split() for t in texts]
        self.bm25 = BM25Okapi(tokenized)
        print(f'Index built: {len(chunks)} chunks.')
    
    def vector_search(self, query: str, top_k: int = 100) -> list[int]:
        q_vec = embed_model.encode([query])[0]
        sims  = np.dot(self.embeddings, q_vec) / (
            np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(q_vec) + 1e-9
        )
        return list(np.argsort(sims)[::-1][:top_k])
    
    def bm25_search(self, query: str, top_k: int = 100) -> list[int]:
        scores = self.bm25.get_scores(query.lower().split())
        return list(np.argsort(scores)[::-1][:top_k])

vector_index = InMemoryVectorIndex()

if all_chunks:
    vector_index.build(all_chunks)
else:
    print('No chunks loaded yet — add documents in Step 1 first.')

In [ ]:
# ── Databricks Vector Search Setup (production) ───────────────
# Skip this cell on Colab — uses Databricks VS API.

if IS_DATABRICKS:
    from databricks.vector_search.client import VectorSearchClient
    
    vsc = VectorSearchClient()
    
    def setup_vs_index():
        """Create Delta-sync Vector Search index (run once)."""
        try:
            vsc.create_delta_sync_index(
                endpoint_name=DB_VS_ENDPOINT,
                index_name=DB_VS_INDEX,
                source_table_name=f'{DB_CATALOG}.{DB_SCHEMA}.documents',
                pipeline_type='TRIGGERED',
                primary_key='chunk_id',
                embedding_source_column='chunk_text',
                embedding_model_endpoint_name='databricks-bge-large-en',
            )
            print(f'VS index created: {DB_VS_INDEX}')
        except Exception as e:
            print(f'Index may already exist: {e}')
    
    def sync_vs_index():
        """Trigger incremental sync after Delta upsert."""
        idx = vsc.get_index(DB_VS_ENDPOINT, DB_VS_INDEX)
        idx.sync()
        print('Vector index sync triggered.')
    
    # setup_vs_index()  # Uncomment on first run
    # sync_vs_index()   # Call after each upsert_chunks()

## 3. Production Retrieval: Hybrid Search + RBAC + Reranking

In [ ]:
from FlagEmbedding import FlagReranker

reranker = FlagReranker(RERANK_MODEL, use_fp16=True)
print(f'Reranker loaded: {RERANK_MODEL}')

def rrf_fusion(ranked_lists: list[list[int]], k: int = 60) -> list[int]:
    """Reciprocal Rank Fusion — merges multiple ranked lists."""
    scores = {}
    for ranked in ranked_lists:
        for rank, idx in enumerate(ranked):
            scores[idx] = scores.get(idx, 0) + 1 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def rbac_filter(chunks: list[dict], user_role: str) -> list[dict]:
    """Filter chunks to only those the user_role is allowed to see."""
    allowed = ROLE_MAP.get(user_role, ['public'])
    filtered = [c for c in chunks if c.get('collection', 'public') in allowed]
    print(f'RBAC: {len(chunks)} -> {len(filtered)} chunks (role={user_role}, allowed={allowed})')
    return filtered

def hybrid_retrieve(query: str, user_role: str, stage1_k: int = 100) -> list[dict]:
    """Stage 1: Hybrid BM25 + Vector Search with RBAC filter."""
    # Get all chunks allowed for this role
    allowed_chunks = rbac_filter(vector_index.chunks, user_role)
    allowed_ids = {i for i, c in enumerate(vector_index.chunks) 
                   if c in allowed_chunks}
    
    # BM25 + Vector over full index, then RBAC-filter
    bm25_ids = [i for i in vector_index.bm25_search(query, top_k=stage1_k)
                if i in allowed_ids]
    vec_ids  = [i for i in vector_index.vector_search(query, top_k=stage1_k)
                if i in allowed_ids]
    
    # Fuse rankings
    fused_ids = rrf_fusion([bm25_ids, vec_ids])[:stage1_k]
    return [vector_index.chunks[i] for i in fused_ids]

def rerank(query: str, candidates: list[dict], top_n: int = 5) -> list[dict]:
    """Stage 2: BGE-v2 cross-encoder reranking."""
    if not candidates:
        return []
    pairs  = [(query, c['chunk_text']) for c in candidates]
    scores = reranker.compute_score(pairs, normalize=True)
    ranked = sorted(zip(scores, candidates), key=lambda x: -x[0])
    result = [{'rerank_score': round(s, 4), **doc} for s, doc in ranked[:top_n]]
    return result

def two_stage_retrieve(query: str, user_role: str, top_n: int = 5) -> list[dict]:
    """Full two-stage: hybrid recall -> rerank."""
    candidates = hybrid_retrieve(query, user_role, stage1_k=100)
    return rerank(query, candidates, top_n=top_n)

print('Retrieval pipeline ready.')

## 4. Production Prompt + LLM Generation

In [ ]:
# Production-grade system prompt (Chapter 3, Slide 4)
SYSTEM_PROMPT = """You are a helpful assistant. Answer ONLY using the provided context below.

Rules:
- Cite the source document for each factual claim using [source: <filename>].
- If the context does not cover the question, say: "I don't have reliable information on this topic."
- Never guess, infer, or go beyond the provided context.
- Respond in the same language as the user's question.
- Maximum response: 3 paragraphs or a short bullet list.
- If multiple sources conflict, note the discrepancy rather than choosing one.
"""

def build_context(chunks: list[dict]) -> str:
    """Format retrieved chunks into LLM context string with source attribution."""
    parts = []
    for i, chunk in enumerate(chunks, 1):
        source = chunk.get('source', 'unknown')
        text   = chunk.get('chunk_text', '')
        score  = chunk.get('rerank_score', '')
        parts.append(f'[{i}] (source: {source}, score: {score})\n{text}')
    return '\n\n---\n\n'.join(parts)

def llm_generate(query: str, chunks: list[dict]) -> str:
    """Call the LLM with the production-grade prompt."""
    context = build_context(chunks)
    user_msg = f'Context:\n{context}\n\nQuestion: {query}'
    
    if LLM_PROVIDER == 'openai':
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': user_msg},
            ],
            temperature=0.0,
            max_tokens=1024,
        )
        return response.choices[0].message.content
    
    elif LLM_PROVIDER == 'local':
        # Ollama or LM Studio running locally
        from openai import OpenAI
        client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
        response = client.chat.completions.create(
            model='llama3.2',
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': user_msg},
            ],
            temperature=0.0,
        )
        return response.choices[0].message.content
    
    else:
        # Placeholder for demo without API key
        return f'[DEMO] Would answer: "{query}" using {len(chunks)} retrieved chunks.'

print('LLM generation configured.')

## 5. Hallucination Detection + Full Production Pipeline

In [ ]:
# Load HHEM hallucination detection model
try:
    from vectara_hhem import HHEM
    hhem_model = HHEM.load()
    HHEM_AVAILABLE = True
    print('HHEM loaded: Vectara hallucination model ready.')
except ImportError:
    HHEM_AVAILABLE = False
    print('HHEM not available — using placeholder score 1.0.')

def score_hallucination(context: str, answer: str) -> float:
    """Returns 0.0 (hallucinated) to 1.0 (fully grounded)."""
    if not HHEM_AVAILABLE or not context.strip() or not answer.strip():
        return 1.0  # Assume grounded if model unavailable
    try:
        return float(hhem_model.predict([(context, answer)])[0])
    except Exception as e:
        print(f'HHEM error: {e}')
        return 1.0

In [ ]:
import time

def production_rag_query(
    query: str,
    user_role: str = 'employee',
    top_n: int = 5,
    max_retries: int = 1,
) -> dict:
    """
    Full production RAG pipeline:
    1. Input safety check (placeholder — add ShieldGemma for production)
    2. Two-stage RBAC retrieval
    3. LLM generation with production prompt
    4. Hallucination detection with HHEM
    5. Retry with wider context if below threshold
    6. Abstain if still failing
    """
    t_start = time.time()
    
    # ── Step 1: Basic input validation ────────────────────────
    query = query.strip()
    if len(query) < 3:
        return {'answer': None, 'blocked': True, 'reason': 'query_too_short',
                'latency_ms': 0}
    
    # ── Step 2: Retrieve ───────────────────────────────────────
    t_ret = time.time()
    chunks = two_stage_retrieve(query, user_role, top_n=top_n)
    retrieve_ms = (time.time() - t_ret) * 1000
    
    if not chunks:
        return {'answer': "I don't have relevant information on this topic.",
                'sources': [], 'hhem_score': 1.0, 'abstained': False,
                'blocked': False, 'retrieve_ms': retrieve_ms,
                'latency_ms': (time.time() - t_start) * 1000}
    
    # ── Step 3: Generate ───────────────────────────────────────
    t_gen = time.time()
    context = build_context(chunks)
    answer  = llm_generate(query, chunks)
    generate_ms = (time.time() - t_gen) * 1000
    
    # ── Step 4: Hallucination check ────────────────────────────
    hhem_score = score_hallucination(context, answer)
    
    # ── Step 5: Retry with wider context if below threshold ────
    retry_count = 0
    while hhem_score < HHEM_PASS_THRESHOLD and retry_count < max_retries:
        retry_count += 1
        print(f'HHEM score {hhem_score:.3f} < {HHEM_PASS_THRESHOLD} — retry {retry_count}')
        chunks  = two_stage_retrieve(query, user_role, top_n=top_n + 4)
        context = build_context(chunks)
        answer  = llm_generate(query, chunks)
        hhem_score = score_hallucination(context, answer)
    
    # ── Step 6: Abstain if still below hard floor ──────────────
    abstained = False
    if hhem_score < HHEM_ABSTAIN_THRESHOLD:
        answer    = "I don't have reliable enough information to answer this accurately."
        abstained = True
    
    total_ms = (time.time() - t_start) * 1000
    
    return {
        'answer':      answer,
        'sources':     list({c.get('source','?') for c in chunks}),
        'hhem_score':  round(hhem_score, 4),
        'abstained':   abstained,
        'blocked':     False,
        'retry_count': retry_count,
        'retrieve_ms': round(retrieve_ms),
        'generate_ms': round(generate_ms),
        'latency_ms':  round(total_ms),
        'num_chunks':  len(chunks),
    }

# ── Quick test ─────────────────────────────────────────────────
if all_chunks:
    result = production_rag_query('What is the refund policy?', user_role='employee')
    print(json.dumps(result, indent=2))
else:
    print('Load documents in Step 1 to test the full pipeline.')

## 6. MLflow Experiment Tracking

In [ ]:
import mlflow

# On Databricks: MLflow is built-in — no setup needed.
# On Colab: uses local file-based tracking.
if not IS_DATABRICKS:
    mlflow.set_tracking_uri('file:///tmp/mlruns')

mlflow.set_experiment('/RAG/Production/Chapter3')

def tracked_query(
    query: str,
    user_role: str = 'employee',
    tags: dict = None,
) -> dict:
    """Run a RAG query and log all metrics to MLflow."""
    with mlflow.start_run(tags=tags or {}):
        mlflow.log_params({
            'query':      query[:250],
            'user_role':  user_role,
            'embed_model': EMBED_MODEL,
            'llm_model':   LLM_MODEL,
        })
        
        result = production_rag_query(query, user_role=user_role)
        
        mlflow.log_metrics({
            'hhem_score':   result.get('hhem_score', 0),
            'retrieve_ms':  result.get('retrieve_ms', 0),
            'generate_ms':  result.get('generate_ms', 0),
            'latency_ms':   result.get('latency_ms', 0),
            'num_chunks':   result.get('num_chunks', 0),
            'abstained':    int(result.get('abstained', False)),
            'retry_count':  result.get('retry_count', 0),
        })
        
        mlflow.log_param('answer_preview', str(result.get('answer', ''))[:200])
        mlflow.log_param('sources', str(result.get('sources', []))[:200])
        
        # Alert on quality/latency issues
        if result.get('latency_ms', 0) > LATENCY_P95_TARGET_MS:
            print(f'⚠️  Latency alert: {result["latency_ms"]}ms > {LATENCY_P95_TARGET_MS}ms target')
            mlflow.set_tag('alert', 'high_latency')
        if result.get('hhem_score', 1.0) < HHEM_PASS_THRESHOLD:
            print(f'⚠️  Quality alert: HHEM {result["hhem_score"]:.3f} < {HHEM_PASS_THRESHOLD}')
            mlflow.set_tag('alert', 'low_quality')
        
        return result

print('MLflow tracking configured.')

## 7. Latency Monitoring + KPI Dashboard

In [ ]:
import pandas as pd
import numpy as np

class LatencyMonitor:
    """Lightweight rolling KPI monitor.
    On Databricks: replace with Structured Streaming on rag_prod.query_logs Delta table.
    """
    def __init__(self, window: int = 100):
        self.log = []
        self.window = window
    
    def record(self, result: dict):
        self.log.append({
            'ts':          pd.Timestamp.now(),
            'latency_ms':  result.get('latency_ms', 0),
            'retrieve_ms': result.get('retrieve_ms', 0),
            'generate_ms': result.get('generate_ms', 0),
            'hhem_score':  result.get('hhem_score', 1.0),
            'abstained':   result.get('abstained', False),
            'num_chunks':  result.get('num_chunks', 0),
        })
    
    def report(self) -> pd.DataFrame:
        if not self.log:
            return pd.DataFrame()
        df = pd.DataFrame(self.log[-self.window:])
        summary = pd.DataFrame([{
            'queries':           len(df),
            'mean_latency_ms':   df['latency_ms'].mean().round(0),
            'p95_latency_ms':    df['latency_ms'].quantile(0.95).round(0),
            'mean_hhem':         df['hhem_score'].mean().round(4),
            'abstain_rate_%':    (df['abstained'].mean() * 100).round(1),
            'mean_chunks':       df['num_chunks'].mean().round(1),
            'p95_ok':            df['latency_ms'].quantile(0.95) <= LATENCY_P95_TARGET_MS,
            'hhem_ok':           df['hhem_score'].mean() >= HHEM_PASS_THRESHOLD,
        }])
        return summary
    
    def plot(self):
        """Plot latency and HHEM trend (requires matplotlib)."""
        try:
            import matplotlib.pyplot as plt
            df = pd.DataFrame(self.log[-self.window:])
            if df.empty: return
            
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
            
            ax1.plot(df.index, df['latency_ms'], label='Total', color='#0071E3')
            ax1.plot(df.index, df['retrieve_ms'], label='Retrieve', color='#30D158', alpha=0.7)
            ax1.plot(df.index, df['generate_ms'], label='Generate', color='#FF9500', alpha=0.7)
            ax1.axhline(LATENCY_P95_TARGET_MS, color='#FF3B30', linestyle='--', label='P95 target')
            ax1.set_title('Query Latency (ms)'); ax1.legend(); ax1.set_xlabel('Query #')
            
            ax2.plot(df.index, df['hhem_score'], color='#BF5AF2', label='HHEM score')
            ax2.axhline(HHEM_PASS_THRESHOLD, color='#FF9500', linestyle='--', label='Pass threshold')
            ax2.axhline(HHEM_ABSTAIN_THRESHOLD, color='#FF3B30', linestyle='--', label='Abstain threshold')
            ax2.set_ylim(0, 1.05); ax2.set_title('Hallucination Score (HHEM)'); ax2.legend()
            ax2.set_xlabel('Query #')
            
            plt.tight_layout(); plt.show()
        except ImportError:
            print('matplotlib not available — install to plot.')

monitor = LatencyMonitor(window=200)
print('Latency monitor ready.')

## 8. A/B Testing Framework for Safe Upgrades

In [ ]:
import random

class ABTestFramework:
    """
    A/B test framework for safe pipeline upgrades (Chapter 3).
    Routes a configurable % of traffic to a 'candidate' pipeline
    while keeping the rest on the 'control' (stable production) pipeline.
    Logs all metrics to MLflow for comparison.
    """
    
    def __init__(self, traffic_split: float = 0.1):
        """
        traffic_split: fraction of queries routed to candidate (default 10%).
        """
        self.traffic_split = traffic_split
        self.control_fn   = None   # stable pipeline function
        self.candidate_fn = None   # new pipeline under test
        self.results = {'control': [], 'candidate': []}
        print(f'A/B test: {int(traffic_split*100)}% traffic to candidate.')
    
    def set_control(self, fn): 
        self.control_fn = fn
    
    def set_candidate(self, fn): 
        self.candidate_fn = fn
    
    def query(self, query: str, user_role: str = 'employee') -> dict:
        use_candidate = random.random() < self.traffic_split
        variant = 'candidate' if use_candidate else 'control'
        fn      = self.candidate_fn if use_candidate else self.control_fn
        
        if fn is None:
            raise ValueError(f'{variant} pipeline not set. Use set_control() / set_candidate().')
        
        with mlflow.start_run(tags={'variant': variant, 'ab_test': 'true'}):
            mlflow.log_params({'query': query[:250], 'user_role': user_role, 'variant': variant})
            result = fn(query, user_role)
            result['variant'] = variant
            mlflow.log_metrics({
                'hhem_score':  result.get('hhem_score', 0),
                'latency_ms':  result.get('latency_ms', 0),
                'abstained':   int(result.get('abstained', False)),
            })
            self.results[variant].append(result)
        return result
    
    def report(self) -> pd.DataFrame:
        """Summarise KPIs for control vs candidate."""
        rows = []
        for variant, results in self.results.items():
            if not results:
                continue
            hhem   = [r.get('hhem_score', 0) for r in results]
            lat    = [r.get('latency_ms', 0) for r in results]
            abst   = [r.get('abstained', False) for r in results]
            rows.append({
                'variant':          variant,
                'n_queries':        len(results),
                'mean_hhem':        round(np.mean(hhem), 4),
                'mean_latency_ms':  round(np.mean(lat)),
                'p95_latency_ms':   round(np.percentile(lat, 95)),
                'abstain_rate_%':   round(np.mean(abst) * 100, 1),
            })
        return pd.DataFrame(rows).set_index('variant')

# ── Example A/B test setup ─────────────────────────────────────
def control_pipeline(q, role):
    return production_rag_query(q, user_role=role, top_n=5)

def candidate_pipeline(q, role):
    # Example: test with top_n=7 (wider context)
    return production_rag_query(q, user_role=role, top_n=7)

ab_test = ABTestFramework(traffic_split=0.2)
ab_test.set_control(control_pipeline)
ab_test.set_candidate(candidate_pipeline)

print('A/B test framework ready.')
print('Usage: ab_test.query("your question", user_role="employee")')
print('       ab_test.report()  — compare KPIs')

## 9. Cost Estimation Utility

In [ ]:
def estimate_monthly_cost(
    daily_queries: int,
    avg_chunks: int = 5,
    avg_chunk_tokens: int = 128,
    avg_answer_tokens: int = 256,
    num_docs: int = 10_000,
    avg_doc_tokens: int = 4096,
) -> pd.DataFrame:
    """
    Rough monthly cost estimate for a production RAG stack.
    Prices approximate as of early 2025 — check current vendor pricing.
    """
    monthly_queries = daily_queries * 30
    
    # Embedding cost (ingest: one-time; query: every request)
    embed_ingest_tokens = num_docs * avg_doc_tokens
    embed_query_tokens  = monthly_queries * 1  # one embed per query
    embed_cost_per_1m   = 0.02  # OpenAI text-embedding-3-small
    embed_ingest_cost   = embed_ingest_tokens / 1e6 * embed_cost_per_1m
    embed_query_cost    = embed_query_tokens  / 1e6 * embed_cost_per_1m * 30
    
    # LLM generation cost
    ctx_tokens       = avg_chunks * avg_chunk_tokens
    llm_in_per_q     = ctx_tokens + 512  # system prompt + context
    llm_out_per_q    = avg_answer_tokens
    llm_in_cost_1m   = 0.15   # GPT-4o-mini input
    llm_out_cost_1m  = 0.60   # GPT-4o-mini output
    llm_cost = monthly_queries * (
        llm_in_per_q  / 1e6 * llm_in_cost_1m +
        llm_out_per_q / 1e6 * llm_out_cost_1m
    )
    
    # Vector DB (Databricks Vector Search approx)
    vs_cost = max(50, num_docs / 100)  # $50 base + $0.01/100 docs/month
    
    # Compute (GPU for reranker — 1 x A10G, ~$1.50/hr)
    gpu_hrs       = (monthly_queries * 0.2) / 3600  # ~200ms per rerank batch
    compute_cost  = max(30, gpu_hrs * 1.50)
    
    # Monitoring & MLflow
    monitor_cost = 20  # rough estimate
    
    total = embed_ingest_cost + embed_query_cost + llm_cost + vs_cost + compute_cost + monitor_cost
    
    df = pd.DataFrame([
        {'Component': 'Embedding (ingest)',       'Cost_USD': round(embed_ingest_cost, 2)},
        {'Component': 'Embedding (queries)',      'Cost_USD': round(embed_query_cost, 2)},
        {'Component': 'LLM generation (GPT-4o-mini)', 'Cost_USD': round(llm_cost, 2)},
        {'Component': 'Vector DB (Databricks VS)', 'Cost_USD': round(vs_cost, 2)},
        {'Component': 'GPU compute (reranker)',   'Cost_USD': round(compute_cost, 2)},
        {'Component': 'Monitoring / MLflow',      'Cost_USD': round(monitor_cost, 2)},
        {'Component': '─── TOTAL ───',            'Cost_USD': round(total, 2)},
    ])
    return df

# Example: 500 queries/day, 10k documents
cost_df = estimate_monthly_cost(daily_queries=500, num_docs=10_000)
print('Estimated Monthly RAG Cost (USD):')
print(cost_df.to_string(index=False))
print('\nNote: 3-5x multiplier for indirect costs (team, security, compliance, HA).')

## 10. Databricks Structured Streaming KPI Monitor
*Run this cell only on Databricks — requires Delta table `rag_prod.query_logs`*

In [ ]:
if IS_DATABRICKS:
    from pyspark.sql.functions import avg, percentile_approx, col, window, count, lit
    from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
    
    # Schema for query logs
    LOG_SCHEMA = StructType([
        StructField('ts',           TimestampType(), True),
        StructField('query',        StringType(),    True),
        StructField('user_role',    StringType(),    True),
        StructField('retrieve_ms',  DoubleType(),    True),
        StructField('generate_ms',  DoubleType(),    True),
        StructField('latency_ms',   DoubleType(),    True),
        StructField('hhem_score',   DoubleType(),    True),
        StructField('abstained',    StringType(),    True),
        StructField('num_chunks',   DoubleType(),    True),
    ])
    
    # Read streaming query logs from Delta
    query_logs_stream = (
        spark.readStream
             .schema(LOG_SCHEMA)
             .format('delta')
             .table(f'{DB_CATALOG}.{DB_SCHEMA}.query_logs')
    )
    
    # Compute rolling 5-minute KPIs
    live_metrics = (
        query_logs_stream
        .withWatermark('ts', '10 minutes')
        .groupBy(window(col('ts'), '5 minutes'))
        .agg(
            count('*').alias('query_count'),
            avg('retrieve_ms').alias('avg_retrieve_ms'),
            avg('generate_ms').alias('avg_generate_ms'),
            percentile_approx('latency_ms', 0.95).alias('p95_latency_ms'),
            avg('hhem_score').alias('avg_hhem_score'),
        )
    )
    
    # Write to Delta for Databricks SQL Alerts dashboard
    streaming_query = (
        live_metrics.writeStream
        .format('delta')
        .outputMode('append')
        .option('checkpointLocation', '/dbfs/rag_prod/checkpoints/live_metrics')
        .table(f'{DB_CATALOG}.{DB_SCHEMA}.live_metrics')
        .start()
    )
    
    print('Streaming KPI monitor started.')
    print(f'Dashboard table: {DB_CATALOG}.{DB_SCHEMA}.live_metrics')
    print('Set Databricks SQL Alerts on: p95_latency_ms > 6000 OR avg_hhem_score < 0.75')
else:
    print('Skipping Databricks Streaming — not in Databricks environment.')
    print('On Databricks: this creates a live_metrics Delta table for SQL Alerts.')
    print('Use monitor.report() and monitor.plot() for local KPI tracking instead.')

---
## Summary — Production RAG Checklist

| Component | Status | Notes |
|-----------|--------|-------|
| docling ingestion | ✅ | Multi-format, structure-aware |
| Delta Lake CDC | ✅ Databricks / 🔄 Parquet fallback | Incremental hash-based upsert |
| Databricks Vector Search | ✅ Databricks only | TRIGGERED sync from Delta |
| Hybrid BM25 + Vector | ✅ | RRF fusion |
| RBAC filtering | ✅ | Role-based collection access |
| BGE-v2 Reranking | ✅ | Two-stage pipeline |
| Production prompt | ✅ | Cite sources + abstain rules |
| HHEM hallucination guard | ✅ | Retry + abstain logic |
| MLflow tracking | ✅ | Every query logged |
| Latency monitor | ✅ | P95 + HHEM rolling KPIs |
| Streaming alerts | ✅ Databricks only | SQL Alerts on live_metrics |
| A/B test framework | ✅ | Safe upgrade with traffic split |
| Cost estimator | ✅ | Monthly TCO calculator |

**Next Steps:**
1. Load your documents in Step 1 (`SAMPLE_DOCS`)
2. Configure your LLM API key (`OPENAI_API_KEY`)
3. Run `tracked_query('your question', user_role='employee')`
4. Check MLflow UI for metrics at `http://localhost:5000` (Colab) or the Databricks MLflow sidebar
5. Set up Databricks SQL Alerts on `live_metrics` for P95 and HHEM thresholds

---
*Chapter 3: Deploying RAG to Production — O'Reilly 2025*